#**Data Loading**

In [19]:
import os
import shutil

In [20]:
!mkdir -p ~/.kaggle

In [21]:
shutil.move("kaggle.json", "/root/.kaggle/")

Error: Destination path '/root/.kaggle/kaggle.json' already exists

In [ ]:
os.chmod("/root/.kaggle/kaggle.json", 600)

In [22]:
from kaggle.api.kaggle_api_extended import KaggleApi

api = KaggleApi()
api.authenticate()
api.dataset_download_files(
  "shashwatwork/knee-osteoarthritis-dataset-with-severity",
  path='/content',
  unzip=True
)

Dataset URL: https://www.kaggle.com/datasets/shashwatwork/knee-osteoarthritis-dataset-with-severity


In [23]:
os.system('rm -rf /content/auto_test')


0

#**Preprocessing**

In [24]:
import numpy as np
import pandas as pd
import os
import time
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Activation,Dropout,Conv2D, MaxPooling2D,BatchNormalization
from tensorflow.keras.optimizers import Adam, Adamax
from tensorflow.keras.metrics import categorical_crossentropy
from tensorflow.keras import regularizers
from tensorflow.keras.models import Model
from tensorflow.keras import backend as K
import time
import shutil
from keras.models import Sequential
from keras import layers
from keras.applications import DenseNet121
from tensorflow.keras.preprocessing import image
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input, Dropout
from tensorflow.keras.optimizers.legacy import Adam, SGD

In [25]:
train_path = '/content/train'
val_path = '/content/val'
test_path = '/content/test'
list_of_classes=['Healthy', 'Doubtful', 'Minimal', 'Moderate', 'Severe']

In [26]:
for d in [train_path, test_path, val_path]:
    filepaths = []
    labels=[]
    classlist=os.listdir(d)
    for klass in classlist:
        intklass=int(klass)
        label=list_of_classes[intklass]
        classpath=os.path.join(d, klass)
        flist=os.listdir(classpath)
        for f in flist:
            fpath=os.path.join(classpath,f)
            filepaths.append(fpath)
            labels.append(label)
    Fseries=pd.Series(filepaths, name='filepaths')
    Lseries=pd.Series(labels, name='labels')
    pdf=pd.concat([Fseries, Lseries], axis=1)
    if d == test_path:
        test_df=pdf
    elif d == val_path:
        valid_df=pdf
    else:
        train_df=pdf

In [27]:
def trim(df, max_samples, min_samples, column):
    df=df.copy()
    groups=df.groupby(column)
    trimmed_df = pd.DataFrame(columns = df.columns)

    for label in df[column].unique():
        group=groups.get_group(label)
        count=len(group)
        if count > max_samples:
            sampled_group=group.sample(n=max_samples, random_state=123,axis=0)
            trimmed_df=pd.concat([trimmed_df, sampled_group], axis=0)
        else:
            if count>=min_samples:
                sampled_group=group
                trimmed_df=pd.concat([trimmed_df, sampled_group], axis=0)
    print('after trimming, the maximum samples in any class is now ',max_samples, ' and the minimum samples in any class is ', min_samples)
    return trimmed_df

max_samples=500 # since each class has more than 200 images all classes will be trimmed to have 200 images per class
min_samples=173
column='labels'
train_df= trim(train_df, max_samples, min_samples, column)

after trimming, the maximum samples in any class is now  500  and the minimum samples in any class is  173


In [28]:
def balance(df, n, working_dir, img_size):
    df=df.copy()
    print('Initial length of dataframe is ', len(df))
    aug_dir=os.path.join(working_dir, 'aug')# directory to store augmented images
    if os.path.isdir(aug_dir):# start with an empty directory
        shutil.rmtree(aug_dir)
    os.mkdir(aug_dir)
    for label in df['labels'].unique():
        dir_path=os.path.join(aug_dir,label)
        os.mkdir(dir_path) # make class directories within aug directory
    # create and store the augmented images
    total=0
    gen=ImageDataGenerator(horizontal_flip=True,  rotation_range=20, width_shift_range=.2,height_shift_range=.2, zoom_range=.2)
    groups=df.groupby('labels') # group by class
    for label in df['labels'].unique():  # for every class
        group=groups.get_group(label)  # a dataframe holding only rows with the specified label
        sample_count=len(group)   # determine how many samples there are in this class
        if sample_count< n: # if the class has less than target number of images
            aug_img_count=0
            delta=n - sample_count  # number of augmented images to create
            target_dir=os.path.join(aug_dir, label)  # define where to write the images
            msg='{0:40s} for class {1:^30s} creating {2:^5s} augmented images'.format(' ', label, str(delta))
            print(msg, '\r', end='')
            aug_gen=gen.flow_from_dataframe(group,  x_col='filepaths', y_col=None, target_size=img_size,
                                            class_mode=None, batch_size=1, shuffle=False,
                                            save_to_dir=target_dir, save_prefix='aug-', color_mode='rgb',
                                            save_format='jpg')
            while aug_img_count<delta:
                images=next(aug_gen)
                aug_img_count += len(images)
            total +=aug_img_count
    print('Total Augmented images created= ', total)
    # create aug_df and merge with train_df to create composite training set ndf
    aug_fpaths=[]
    aug_labels=[]
    classlist=os.listdir(aug_dir)
    for klass in classlist:
        classpath=os.path.join(aug_dir, klass)
        flist=os.listdir(classpath)
        for f in flist:
          fpath=os.path.join(classpath,f)
          aug_fpaths.append(fpath)
          aug_labels.append(klass)
    Fseries=pd.Series(aug_fpaths, name='filepaths')
    Lseries=pd.Series(aug_labels, name='labels')
    aug_df=pd.concat([Fseries, Lseries], axis=1)
    df=pd.concat([df,aug_df], axis=0).reset_index(drop=True)
    print('Length of augmented dataframe is now ', len(df))
    return df

n=500 # number of samples in each class
working_dir=r'./' # directory to store augmented images
img_size=(224,224) # size of augmented images
train_df=balance(train_df, n, working_dir, img_size)

Initial length of dataframe is  2173
Found 173 validated image filenames.
Total Augmented images created=  327
Length of augmented dataframe is now  2500


In [29]:
def relabel(label_in):
    if label_in =='Healthy' or label_in == 'Doubtful' or label_in =='Minimal':
        label='Healthy'
    else:
        label=label_in
    return label


for df in [train_df, test_df, valid_df]:
    for i in range(len(df)):
        label=df['labels'].iloc[i]
        new_label=relabel(label)
        df['labels'].iloc[i]=new_label
print(train_df['labels'].value_counts())

<ipython-input-29-99e949121a31>:13: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  df['labels'].iloc[i]=new_label


labels
Healthy     1500
Moderate     500
Severe       500
Name: count, dtype: int64


In [30]:
batch_size=20 # We will use and EfficientetB3 model, with image size of (200, 250) this size should not cause resource error
trgen=ImageDataGenerator(horizontal_flip=True,rotation_range=20 )
t_and_v_gen=ImageDataGenerator()
msg='{0:70s} for train generator'.format(' ')
print(msg, '\r', end='') # prints over on the same line
train_gen=trgen.flow_from_dataframe(train_df, x_col='filepaths', y_col='labels', target_size=img_size,
                                   class_mode='categorical', color_mode='rgb', shuffle=True, batch_size=batch_size)
msg='{0:70s} for valid generator'.format(' ')
print(msg, '\r', end='') # prints over on the same line
valid_gen=t_and_v_gen.flow_from_dataframe(valid_df, x_col='filepaths', y_col='labels', target_size=img_size,
                                   class_mode='categorical', color_mode='rgb', shuffle=False, batch_size=batch_size)
# for the test_gen we want to calculate the batch size and test steps such that batch_size X test_steps= number of samples in test set
# this insures that we go through all the sample in the test set exactly once.
length=len(test_df)
test_batch_size=sorted([int(length/n) for n in range(1,length+1) if length % n ==0 and length/n<=80],reverse=True)[0]
test_steps=int(length/test_batch_size)
msg='{0:70s} for test generator'.format(' ')
print(msg, '\r', end='') # prints over on the same line
test_gen=t_and_v_gen.flow_from_dataframe(test_df, x_col='filepaths', y_col='labels', target_size=img_size,
                                   class_mode='categorical', color_mode='rgb', shuffle=False, batch_size=test_batch_size)
# from the generator we can get information we will need later
classes=list(train_gen.class_indices.keys())
print(classes)
class_indices=list(train_gen.class_indices.values())
class_count=len(classes)
labels=test_gen.labels
print('test batch size: ' ,test_batch_size, '  test steps: ', test_steps, ' number of classes : ', class_count)

Found 2500 validated image filenames belonging to 3 classes.
Found 826 validated image filenames belonging to 3 classes.
Found 1656 validated image filenames belonging to 3 classes.
['Healthy', 'Moderate', 'Severe']
test batch size:  72   test steps:  23  number of classes :  3


#**Model Architecture**

In [34]:
input_tensor = Input(shape=(224, 224, 3))
base_model = DenseNet121(weights='imagenet',
                         include_top=False,
                         input_shape=(224,224,3),
                         pooling='max')

base_model.trainable=True

x = base_model.output

x=BatchNormalization(axis=-1, momentum=0.99, epsilon=0.001)(x)

x = Dense(256,
          kernel_regularizer=regularizers.l2(0.016),  # Fixed here
          activity_regularizer=regularizers.l1(0.006),  # Fixed here
          bias_regularizer=regularizers.l1(0.006),  # Fixed here
          activation='relu')(x)
x=Dropout(rate=.4, seed=123)(x)

output=Dense(3, activation='linear')(x)

model=Model(inputs=base_model.input, outputs=output)

#**CallBacks**

In [35]:
#Saves the most with best performance among all epochs
checkpoint_path = "/content/best_model.keras"
checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    monitor='val_categorical_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

In [36]:
#Reduces learning rate when there is no increase in accuracy
learning_rate_callback = tf.keras.callbacks.ReduceLROnPlateau(
  monitor='val_categorical_accuracy',
  factor=0.3,
  patience=3,
  min_lr=1e-6,
  min_delta=0.01,
)

#**Training**

In [37]:
from keras.losses import CategoricalCrossentropy
from keras.optimizers import Adam
from keras.metrics import CategoricalAccuracy

model.compile(optimizer = Adam(0.001),
              loss = CategoricalCrossentropy(from_logits=True),
              metrics=[CategoricalAccuracy()])

In [38]:
#The number of layers and their hyperparameters are adjusted to maximize the validation accuracy.
history = model.fit(train_gen, validation_data = valid_gen, epochs = 20, verbose = 1, callbacks = [checkpoint_callback, learning_rate_callback])

/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step - categorical_accuracy: 0.5490 - loss: 9.8328
Epoch 1: val_categorical_accuracy improved from -inf to 0.76634, saving model to /content/best_model.keras
125/125 ━━━━━━━━━━━━━━━━━━━━ 217s 497ms/step - categorical_accuracy: 0.5493 - loss: 9.8045 - val_categorical_accuracy: 0.7663 - val_loss: 7.2598 - learning_rate: 0.0010
Epoch 2/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step - categorical_accuracy: 0.5996 - loss: 3.0369
Epoch 2: val_categorical_accuracy improved from 0.76634 to 0.83898, saving model to /content/best_model.keras
125/125 ━━━━━━━━━━━━━━━━━━━━ 42s 337ms/step - categorical_accuracy: 0.5996 - loss: 3.0342 - val_categorical_accuracy: 0.8390 - val_loss: 3.3289 - learning_rate: 0.0010
Epoch 3/20
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 306ms/step - categorical_accuracy: 0.6046 - loss: 1.8610
Epoch 3: val_categorical_accuracy did not improve from 0.83898
125/125 ━━━━━━━━━━━━━━━━━━━━ 80s 323ms/step - categorical_accuracy: 0.6046 - loss: 

Training with 5 classes gives the accuracy of 40%

Training with 3 classes gives the accuracy of 84%